In [27]:
from torch import tensor
from torchvision import datasets, transforms
import numpy as np
import sklearn
from sklearn.model_selection import StratifiedKFold
import logging
import math
import random
from sklearn.mixture import GaussianMixture
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
from scipy.stats import kstest, anderson_ksamp, cumfreq, ks_2samp, cramervonmises, chisquare, entropy, wasserstein_distance

In [28]:
from keras.preprocessing.image import ImageDataGenerator

In [29]:
def get_data(dataset_type, path=None):
    
    if dataset_type == "MNIST":
        train: datasets = datasets.MNIST(root=path, train=True, download=True, transform=None)
        test: datasets = datasets.MNIST(root=path, train=False, download=True, transform=None)
    elif dataset_type == "CIFAR10":
        transform = transforms.Compose([
            transforms.ToTensor(),
        ])
        train = datasets.CIFAR10(root=path, train=True, download=False)
        train.transform = transform
        train.data = tensor(train.data)
        train.targets = tensor(train.targets)
        test = datasets.CIFAR10(root=path, train=False, download=False)
        test.transform = transform
        test.data = tensor(test.data)
        test.targets = tensor(test.targets)

    return train, test

In [30]:
def log_class_counts(y_train, subset_ID_map, log=False):
    cls_counts = {}

    for subset_i, ID in subset_ID_map.items():
        unq, unq_cnt = np.unique(y_train[ID], return_counts=True)
        tmp = {unq[i]: unq_cnt[i] for i in range(len(unq))}
        cls_counts[subset_i] = tmp

    if log:
        logging.debug('Label distributions: %s' % str(cls_counts))

    return cls_counts

In [31]:
def map_to_prob(y_train, subset_map):
    counts = log_class_counts(y_train, subset_map)

    values = [np.array([counts.get(k).get(key) for key in counts.get(k)]) for k in counts]
    probs = [values[j] / values[j].sum() for j in range(len(values))]

    return probs

In [32]:
def partition_homo_skf(train, n_clients, alpha=0):
    subset_ID_map = {}
    x_train, y_train = train.data, train.targets
    # n_train = y_train.shape[0]
    skf = StratifiedKFold(n_splits=n_clients, shuffle=True, random_state=42)
    subsets = []
    for train_ID, test_ID in skf.split(train, y_train):
        subsets.append(test_ID)
    for j in range(n_clients):
        # np.random.shuffle(subsets[j])
        subset_ID_map[j] = subsets[j]
    return subset_ID_map

In [33]:
def partition_hetero_dir(train, n_clients, alpha):
    x_train, y_train = train.data, train.targets
    
    min_size = 0
    # classes
    K = len(set(y_train.tolist()))
    # data points
    N = y_train.shape[0]
    subset_ID_map = {}

    while min_size < 10:
        subset_ID_list = [[] for _ in range(n_clients)]
        for k in range(K):
            ids_k = np.where(y_train == k)[0]
            np.random.shuffle(ids_k)
            proportions = np.random.dirichlet(np.repeat(alpha, n_clients))
            proportions = np.array(
                [p * (len(ids_j) < N / n_clients) for p, ids_j in zip(proportions, subset_ID_list)])
            proportions = proportions / proportions.sum()
            proportions = (np.cumsum(proportions) * len(ids_k)).astype(int)[:-1]
            subset_ID_list = [ids_j + ids.tolist() for ids_j, ids in
                              zip(subset_ID_list, np.split(ids_k, proportions))]
            min_size = min([len(ids_j) for ids_j in subset_ID_list])

    for j in range(n_clients):
        np.random.shuffle(subset_ID_list[j])
        subset_ID_map[j] = subset_ID_list[j]

    return subset_ID_map

In [34]:
def partition_quantity_based(train, n_clients, alpha):
    # sorted = pd.concat([y for x, y in data.groupby(0)]).reset_index().drop(columns=['index'])
    x, y = train.data, np.array(train.targets)
    K = len(set(y.tolist()))
    # subsets_pure = {i: np.where(y == i)[0] for i in range(10)}
    subsets_pure = np.concatenate([np.where(y == i)[0] for i in range(K)])

    ids = np.arange(train.targets.shape[0])
    batch_ids = np.array_split(ids, n_clients*alpha)
    minibatches = []

    for i in range(n_clients*alpha):
        batch_i = [subsets_pure[j] for j in batch_ids[i]]
        minibatches.append(batch_i)

    minibatches = np.array(minibatches)

    ids_mini = np.random.permutation(range(n_clients*alpha))
    subset_indices = np.array_split(ids_mini, n_clients)

    clients = {}

    for index in subset_indices:
        client_i = []
        for i in index:
            client_i.append(minibatches[i])
        client_i = np.concatenate(client_i)
        clients.update({i:np.array(client_i)})

    return clients

In [45]:
def partition(dataset_type, type, n_clients, alpha):
    partition_funcs = {
        "homo" : partition_homo_skf,
        "hetero-dir": partition_hetero_dir,
        #"hetero-gaussian": partition_hetero_gaussian,
        "quant": partition_quantity_based
    }
    
    try:
        partition_func = partition_funcs[type]
    except KeyError:
        raise ValueError(f"Invalid mode: {type}")

    return partition_func(dataset_type, n_clients, alpha)

In [36]:
def tensor_to_csv(x_train, y_train, subset_map, epoch_num):
    for key in subset_map:
        x, y = x_train[subset_map.get(key)], y_train[subset_map.get(key)]
        df_x = pd.DataFrame(x.tolist())
        df_y = pd.DataFrame(y.tolist())
        df_x['targets'] = df_y
        df_x.rename(columns={0: 'data', "targets": "targets"})
        df_x.to_csv(f'subsets/epoch_{epoch_num}_subset_{key + 1}.csv', index=False)

In [40]:
def vis_divergence(evol_stat, mode_plot, main_title, xlab): 
    
    def linplot(stats):
        plt.figure(figsize=(5, 3), dpi=300)
        plt.plot(np.arange(len(stats)), stats)
        plt.xlabel(xlab)
        plt.suptitle(main_title, fontsize='small')
        plt.show()
    
    mode_dict = {
        "lp": linplot
    }
    
    try:
        plot_func = mode_dict[mode_plot]
    except KeyError:
        raise ValueError(f"Invalid mode: {mode}")  
    
    plot_func(evol_stat)

In [43]:
def run_experiment(dataset_type, alpha_vector, path, mode_part, mode_test, mode_plot, n_clients, plot=False, logpath=None, save=False):
    train, test = get_data(dataset_type, path)
    x_train, y_train, x_test, y_test = train.data, train.targets, test.data, test.targets
    evol_stat, evol_pval = [], []

    for j in range(len(alpha_vector)):
        a_j = alpha_vector[j]
        subset_map = partition(train, mode_part, n_clients, a_j)
        if save:
            tensor_to_csv(x_train, y_train, subset_map, j)
        stats, pvals = distance(y_train, subset_map, mode_test)
        mean_teststat_j = np.mean(stats)
        mean_pval_j = np.mean(pvals)
        evol_stat.append(mean_teststat_j)
        evol_pval.append(mean_pval_j)

    title_formats_part = {
        "hetero-dir": "Mean divergence from original distribution under heterogeneous partitioning via Dirichlet distribution",
        #"hetero-gaussian": "Mean divergence from original distribution under heterogeneous partitioning via Gaussian distribution",
        "homo": "A homogeneous partitioning with {n_clients} subsets",
        "quant": "A quantity-based heterogeneous partitioning with {n_clients} subsets"
    }

    title_formats_test = {
        "kolmogorov-smirnov": "Test Statistic: Kolmogorov-Smirnov",
        "empirical": "Test Statistic: Empirical Distribution",
        "kl": "Test Statistic: Kullback-Leibler Divergence (Entropy-Based)",
        "js": "Test Statistic: Jensen-Shannon Divergence (Entropy-Based)",
        "wd": "Test Statistic: Wasserstein Distance",
        "gini": "Test Statistic: Gini Coefficient"
    }
    
    main_title = title_formats_part.get(mode_part, "")
    main_title = main_title.format(n_clients=n_clients)
    appendage = title_formats_test.get(mode_test, "")
    main_title = dataset_type +': ' + main_title + '\n' + appendage
    
    if mode_part == "quant":
        l, u = tuple(round(x,1) for x in (min(alpha_vector), max(alpha_vector)))
        xlab = f'[{u}:{l}], set of divisors for {u}' 
    elif len(alpha_vector) <=15:
        xlab = [f'α_{j}={alpha}' for j, alpha in enumerate(alpha_vector)]
    else:
        l, u, steps = tuple(round(x,1) for x in (min(alpha_vector), max(alpha_vector), (max(alpha_vector)-min(alpha_vector))/(len(alpha_vector)-1)))
        xlab = f'[{l}:{u}], step size = {steps}'

    if plot:
        vis_divergence(evol_stat, mode_plot, main_title, xlab)
    
    return (evol_stat, evol_pval)

In [38]:
def augment(method, x_subset, y_subset, batch_size):
    
    rotation_range_val = 30
    width_shift_range = 0.25
    height_shift_range = 0.25
    shear_range_val = 45
    zoom_range_val = [0.5, 1.5]
    
    methods = {
        "rotate": ImageDataGenerator(rotation_range=rotation_range_val),
        "shift": ImageDataGenerator(width_shift_range=width_shift_range, height_shift_range=height_shift_range),
        "shear": ImageDataGenerator(shear_range=shear_range_val),
        "zoom": ImageDataGenerator(zoom_range=zoom_range_val)
    }

    data_gen = methods[method]
    data_gen.fit(x_subset.reshape(x_subset.shape[0], 28, 28, 1))
    iterator = data_gen.flow(x_subset.reshape(x_subset.shape[0], 28, 28, 1), y_subset.reshape(y_subset.shape[0], 1), batch_size=batch_size, shuffle=False)
    
    return iterator

In [39]:
def getDivs(N):
    factors = {1}
    maxP  = int(N**0.5)
    p,inc = 2,1
    while p <= maxP:
        while N%p==0:
            factors.update([f*p for f in factors])
            N //= p
            maxP = int(N**0.5)
        p,inc = p+inc,2
    if N>1:
        factors.update([f*N for f in factors])
    return sorted(factors)  

In [46]:
run_experiment("MNIST", np.arange(0.5, 100.5, 0.5).tolist(), "D:/", "hetero-dir", "kl", "lp", 10)

NameError: name 'distance' is not defined

In [ ]:
subset_map = partition(train, mode_part, n_clients, a_j)
def augment(subset_map, dataset_type):
    train, test = get_data(dataset_type)
    l = []
    for ID_arr in subset_map.values():
        l.append(train.data[ID_arr])
    
    # TODO: refine transforms
    training_transforms = torchvision.transforms.Compose([
        torchvision.transforms.Resize(size=(32, 32)),
        torchvision.transforms.RandomCrop(size=(28, 28)),
        torchvision.transforms.RandomRotation(degrees=30, interpolation=PIL.Image.BILINEAR),
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize(mean=(0.5,), std=(0.5,)),
        # normalize does (x_i - mean) / std
        # if images are [0, 1], they will be [-1, 1] afterwards
    ])
    
    train_dataset = datasets.MNIST(root='data', 
                               train=True, 
                               transform=training_transforms,
                               download=True)